# Spot (현물) Market Stream PoC — `stream.binance.com`

실제 코인을 사고파는 현물 시장의 공개 WebSocket 스트림을 종류별로 직접 호출하고 원본 payload를 확인한다. API 키는 필요 없다.

- 도메인: `wss://stream.binance.com:9443`
- 예시 심볼: `BTCUSDT`, `ETHUSDT` (아래 `SYMBOL` 변수만 바꾸면 된다)
- 관련 문서: [docs/binanceWebSocketREADME.md](../docs/binanceWebSocketREADME.md)


In [ ]:
%pip install -q "websockets>=17,<18"

In [ ]:
import asyncio
import json

from websockets.asyncio.client import connect

BASE_URL = "wss://stream.binance.com:9443/ws"
SYMBOL = "btcusdt"

In [ ]:
async def receive(stream: str, message_count: int = 3) -> list[dict]:
    url = f"{BASE_URL}/{stream}"
    messages = []

    async with connect(url, ping_interval=20, ping_timeout=60, close_timeout=5) as ws:
        for _ in range(message_count):
            try:
                message = await asyncio.wait_for(ws.recv(), timeout=15)
            except TimeoutError:
                print(f"15초 동안 추가 메시지 없음: {len(messages)}건 수신")
                break
            messages.append(json.loads(message))

    return messages

## 1. Trade Stream

개별 체결이 발생할 때마다 실시간으로 전달된다. 하나의 buyer와 seller 사이에서 발생한 실제 체결 단위다.

- 스트림: `<symbol>@trade`
- 주요 필드: `t` 체결 ID, `p` 가격, `q` 수량, `T` 체결 시간, `m` buyer가 maker인지 여부


In [ ]:
trade_messages = await receive(f"{SYMBOL}@trade", message_count=3)
trade_messages

## 2. Aggregate Trade Stream

하나의 taker 주문에서 발생한 체결들을 집계해서 전달한다. 개별 체결보다 메시지 수를 줄여 거래 흐름을 보고 싶을 때 사용할 수 있다.

- 스트림: `<symbol>@aggTrade`
- 주요 필드: `a` 집계 체결 ID, `p` 가격, `q` 수량, `f` 첫 체결 ID, `l` 마지막 체결 ID, `T` 체결 시간


In [ ]:
aggregate_trade_messages = await receive(f"{SYMBOL}@aggTrade", message_count=3)
aggregate_trade_messages

## 3. Kline Stream

지정한 시간 간격의 캔들 정보를 전달한다. 현재 진행 중인 캔들도 계속 갱신되며 `k.x`가 `true`이면 해당 캔들이 종료된 것이다.

- 스트림: `<symbol>@kline_<interval>`
- 예시 주기: `1s`, `1m`, `5m`, `1h`, `1d`
- 주요 필드: `k.t` 시작 시간, `k.T` 종료 시간, `k.o/h/l/c` OHLC, `k.v` 거래량, `k.x` 종료 여부


In [ ]:
KLINE_INTERVAL = "1m"

kline_messages = await receive(
    f"{SYMBOL}@kline_{KLINE_INTERVAL}",
    message_count=3,
)
kline_messages

## 4. Individual Symbol Book Ticker Stream

현재 최우선 매수호가와 최우선 매도호가가 변경될 때 전달된다. 전체 호가창 없이 best bid/ask와 스프레드만 확인할 때 적합하다.

- 스트림: `<symbol>@bookTicker`
- 주요 필드: `u` update ID, `b` best bid 가격, `B` bid 수량, `a` best ask 가격, `A` ask 수량
- 이 payload에는 이벤트 타입 필드 `e`가 없다.


In [ ]:
book_ticker_messages = await receive(f"{SYMBOL}@bookTicker", message_count=3)
book_ticker_messages

## 5. Diff. Depth Stream

호가창에서 변경된 bid와 ask를 실시간으로 전달한다. 이 메시지만으로 전체 order book을 만들면 안 되며, 운영 구현에서는 REST depth snapshot과 update ID를 기준으로 동기화해야 한다.

- 스트림: `<symbol>@depth` 또는 `<symbol>@depth@100ms`
- 주요 필드: `U` 첫 update ID, `u` 마지막 update ID, `b` 변경된 bids, `a` 변경된 asks


In [ ]:
depth_messages = await receive(f"{SYMBOL}@depth@100ms", message_count=3)
depth_messages

## 6. Partial Book Depth Stream

호가창 전체가 아니라 상위 5, 10, 20단계의 bid/ask를 전달한다. 간단한 호가 비교 PoC에 사용하기 쉽다.

- 스트림: `<symbol>@depth<levels>` 또는 `<symbol>@depth<levels>@100ms`
- 지원 단계: `5`, `10`, `20`
- 주요 필드: `lastUpdateId`, `bids`, `asks`


In [ ]:
partial_depth_messages = await receive(
    f"{SYMBOL}@depth5@100ms",
    message_count=3,
)
partial_depth_messages

## 7. Individual Symbol Ticker Stream

심볼의 최근 24시간 가격 변화와 거래 통계를 전달한다. 현재가, 고가, 저가, 거래량 등 시장 요약을 확인할 수 있다.

- 스트림: `<symbol>@ticker`
- 주요 필드: `c` 현재가, `p` 가격 변화, `P` 변화율, `h` 고가, `l` 저가, `v` base 거래량, `q` quote 거래량


In [ ]:
ticker_messages = await receive(f"{SYMBOL}@ticker", message_count=3)
ticker_messages

## 8. Mini Ticker Stream

24시간 ticker에서 일부 필드만 제공하는 간소화된 스트림이다. 가격과 거래량 중심의 가벼운 모니터링에 사용할 수 있다.

- 스트림: `<symbol>@miniTicker`
- 주요 필드: `c` 종가, `o` 시가, `h` 고가, `l` 저가, `v` base 거래량, `q` quote 거래량


In [ ]:
mini_ticker_messages = await receive(f"{SYMBOL}@miniTicker", message_count=3)
mini_ticker_messages

## 9. Combined Streams

여러 스트림을 WebSocket 하나로 묶어서 수신한다. 각 메시지는 `stream`과 `data` 필드로 감싸져 어느 스트림에서 왔는지 구분할 수 있다.

- URL: `/stream?streams=<stream1>/<stream2>/...`
- 응답 형태: `{"stream": "btcusdt@trade", "data": {...}}`


In [ ]:
async def receive_combined(streams: list[str], message_count: int = 5) -> list[dict]:
    stream_names = "/".join(streams)
    url = f"wss://stream.binance.com:9443/stream?streams={stream_names}"
    messages = []

    async with connect(url, ping_interval=20, ping_timeout=60, close_timeout=5) as ws:
        for _ in range(message_count):
            messages.append(json.loads(await ws.recv()))

    return messages


combined_messages = await receive_combined(
    [f"{SYMBOL}@trade", f"{SYMBOL}@kline_1m", f"{SYMBOL}@bookTicker"],
    message_count=5,
)
combined_messages